# 04 - Hyperparameter Search

Random search co kiem soat de tim config tot nhat.

In [1]:
import random
import sys
from pathlib import Path

import torch
from torch import nn
from torch.optim import AdamW

ROOT = Path('..').resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.append(str(ROOT / 'src'))

from flowers102.data import build_dataloaders
from flowers102.models import create_model
from flowers102.train import fit, TrainConfig
from flowers102.utils import set_seed, save_json

set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [2]:
search_space = {
    'model_name': ['resnet18', 'efficientnet_b0'],
    'lr': [1e-4, 3e-4, 5e-4],
    'weight_decay': [1e-5, 1e-4],
    'batch_size': [16, 32],
    'label_smoothing': [0.0, 0.05, 0.1],
}

def sample_cfg():
    return {k: random.choice(v) for k, v in search_space.items()}

trials = 8
results = []
for t in range(trials):
    cfg = sample_cfg()
    train_loader, valid_loader, _, train_ds, _, _ = build_dataloaders(
        data_dir="/home/nguyenhuynh/Documents/deep-learning/dataset/flower_data",
        batch_size=cfg['batch_size'],
        num_workers=0,
        image_size=224,
        use_augmentation=True,
    )
    model = create_model(cfg['model_name'], num_classes=len(train_ds.classes), pretrained=True)
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg['label_smoothing'])
    optimizer = AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    train_cfg = TrainConfig(epochs=4, device=device, checkpoint_path=str(ROOT / 'checkpoints' / f'search_trial_{t}.pth'))
    hist = fit(model, train_loader, valid_loader, criterion, optimizer, scheduler=None, config=train_cfg)
    score = hist['valid_top1'][-1]
    results.append({'trial': t, **cfg, 'valid_top1_last': score})
    print('trial', t, 'valid_top1_last', round(score, 4))

[Epoch 1/4]  train_loss=2.3304  train_top1=0.5341  valid_loss=0.6388  valid_top1=0.8936  valid_top5=0.9841  lr=0.000100
[Epoch 2/4]  train_loss=0.8760  train_top1=0.8390  valid_loss=0.2807  valid_top1=0.9572  valid_top5=0.9890  lr=0.000100
[Epoch 3/4]  train_loss=0.5737  train_top1=0.8906  valid_loss=0.1780  valid_top1=0.9658  valid_top5=0.9951  lr=0.000100
[Epoch 4/4]  train_loss=0.4289  train_top1=0.9114  valid_loss=0.1410  valid_top1=0.9731  valid_top5=0.9963  lr=0.000100
trial 0 valid_top1_last 0.9731
[Epoch 1/4]  train_loss=2.5154  train_top1=0.5239  valid_loss=1.7381  valid_top1=0.7592  valid_top5=0.9487  lr=0.000500
[Epoch 2/4]  train_loss=1.7703  train_top1=0.7305  valid_loss=1.5637  valid_top1=0.8240  valid_top5=0.9511  lr=0.000500
[Epoch 3/4]  train_loss=1.5994  train_top1=0.7880  valid_loss=1.4468  valid_top1=0.8716  valid_top5=0.9768  lr=0.000500
[Epoch 4/4]  train_loss=1.4789  train_top1=0.8292  valid_loss=1.2494  valid_top1=0.9254  valid_top5=0.9878  lr=0.000500
trial 1 v

In [3]:
import pandas as pd

result_df = pd.DataFrame(results).sort_values('valid_top1_last', ascending=False).reset_index(drop=True)
result_df.head(10)

,trial,model_name,lr,weight_decay,batch_size,label_smoothing,valid_top1_last
0,2,efficientnet_b0,0.0001,0.00001,16,0.00,0.977995
1,4,efficientnet_b0,0.0001,0.00010,32,0.00,0.976773
2,6,resnet18,0.0001,0.00010,16,0.00,0.974328
3,0,resnet18,0.0001,0.00010,16,0.00,0.973105
4,7,efficientnet_b0,0.0001,0.00010,32,0.10,0.962103
5,1,resnet18,0.0005,0.00001,16,0.10,0.925428
6,5,resnet18,0.0005,0.00010,32,0.05,0.910758
7,3,resnet18,0.0005,0.00001,16,0.10,0.904645


In [4]:
best_cfg = result_df.iloc[0].to_dict()
save_json({'results': results, 'best_config': best_cfg}, ROOT / 'reports' / 'hparam_search_results.json')
print('saved:', (ROOT / 'reports' / 'hparam_search_results.json').resolve())
best_cfg

saved: /home/nguyenhuynh/Documents/deep-learning/quangbinh/cv/reports/hparam_search_results.json


{'trial': 2,
 'model_name': 'efficientnet_b0',
 'lr': 0.0001,
 'weight_decay': 1e-05,
 'batch_size': 16,
 'label_smoothing': 0.0,
 'valid_top1_last': 0.9779951100244498}